In [ ]:
import glob
import numpy as np, pandas as pd, geopandas as gpd, folium, osmnx as ox
from sklearn.neighbors import BallTree
from saajha.clean import clean_positions

R = 6371000  # Earth radius in metres
stops = pd.read_csv("../data/raw/delhi_bus_gtfs/stops.txt")
f = sorted(glob.glob("../data/raw/bus_positions_*.csv"))[-1]
pos = clean_positions(pd.read_csv(f, dtype={"route_id": str}))
hours = (pos.timestamp.max() - pos.timestamp.min()) / 3600
print(len(stops), "bus stops |", pos.vehicle_id.nunique(), "buses over", round(hours, 2), "hours")

In [ ]:
ox.settings.overpass_url = "https://overpass.kumi.systems/api"

In [ ]:
ox.settings.requests_timeout = 300
osm = ox.features_from_point((28.61, 77.21), tags={"station": "subway"}, dist=30000)
print(len(osm), "metro stations from OSM")

In [ ]:
raw = pd.read_csv("../data/raw/DELHI_METRO_DATA.csv")
stations = (raw.rename(columns={"Station": "name", "Latitude": "lat", "Longitude": "lon"})
               .dropna(subset=["lat", "lon"])
               .groupby("name", as_index=False)[["lat", "lon"]].mean())
print(len(raw), "rows ->", len(stations), "stations with coordinates")

delhi = ox.geocode_to_gdf("Delhi, India").geometry.iloc[0]
inside = gpd.GeoSeries(gpd.points_from_xy(stations.lon, stations.lat)).within(delhi).values
stations = stations[inside].reset_index(drop=True)
print(len(stations), "metro stations inside Delhi")

In [ ]:
st_xy = np.radians(stations[["lat", "lon"]].values)

stop_tree = BallTree(np.radians(stops[["stop_lat", "stop_lon"]].values), metric="haversine")
stations["bus_stops_500m"] = stop_tree.query_radius(st_xy, r=500 / R, count_only=True)

pt_tree = BallTree(np.radians(pos[["lat", "lon"]].values), metric="haversine")
idx = pt_tree.query_radius(st_xy, r=300 / R)
vids, rids = pos.vehicle_id.values, pos.route_id.values
stations["buses_per_hour"] = [len(set(vids[i])) / hours for i in idx]
stations["live_routes"] = [pd.Series(rids[i]).nunique() for i in idx]
stations.describe()

In [ ]:
worst = stations.sort_values(["buses_per_hour", "bus_stops_500m"]).head(20)
worst

In [ ]:
dup = stations.duplicated(["lat", "lon"], keep=False)
print("dropping (copied coordinates):", stations.loc[dup, "name"].tolist())
stations = stations[~dup].reset_index(drop=True)

In [ ]:
from saajha.geo import haversine_km

metro_stops = stops[stops.stop_name.str.contains("metro", case=False, na=False)]
def check_m(row):
    key = row["name"].replace("†", "").strip().lower()
    m = metro_stops[metro_stops.stop_name.str.lower().str.contains(key, regex=False)]
    return np.nan if m.empty else haversine_km(row.lat, row.lon, m.stop_lat, m.stop_lon).min() * 1000

stations["check_m"] = stations.apply(check_m, axis=1)
bad = stations.check_m > 800
print(f"{bad.sum()} stations look misplaced (named bus stop > 800 m away):")
print(stations.loc[bad, ["name", "check_m"]].round(0).to_string(index=False))
stations = stations[~bad].reset_index(drop=True)
print(len(stations), "stations kept")

In [ ]:
worst = stations.sort_values(["buses_per_hour", "bus_stops_500m"]).head(20)
worst

In [ ]:
for _, s in worst.iterrows():
    print(f"{s['name']}: https://www.google.com/maps?q={s.lat},{s.lon}")

In [ ]:
fixes = {
    "Chawri Bazar":         (28.649600, 77.226200),
    "Shalimar Bagh":        (28.701819, 77.165191),
    "Chirag Delhi":         (28.541229, 77.229385),
    "Naraina Vihar":        (28.624125, 77.136488),
    "Kirti Nagar":          (28.655100, 77.151500),
    "Munirka":              (28.557898, 77.173985),
    "Rajouri Garden":       (28.649000, 77.122600),
    "Mayur Vihar -I":       (28.604134, 77.289526),
    "Mayur Vihar Pocket I": (28.606598, 77.296326),
    "Sector 28":  None,   # Faridabad, not Delhi
    "Shyam Park": None,   # Ghaziabad
    "Vaishali":   None,   # Ghaziabad
}

clean_name = lambda s: s.str.replace("†", "", regex=False).str.strip()
for name, coord in fixes.items():
    hit = clean_name(stations.name) == name
    if not hit.any():
        print("not found:", name)
    elif coord is None:
        stations = stations[~hit]
    else:
        stations.loc[hit, ["lat", "lon"]] = coord
stations = stations.reset_index(drop=True)
print(len(stations), "stations after manual fixes")